In [0]:
#  Re-register all tables — run before any notebook 

STORAGE_ACCOUNT = "adlssupplychain"
CONTAINER       = "supplychain-data"
BASE_ABFSS      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set(
    "spark.databricks.delta.overwriteSchema.enabled",  "true")

# Re-create database with explicit ADLS location
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS supply_chain_db
    LOCATION '{BASE_ABFSS}/hive-warehouse/supply_chain_db.db'
""")

# All tables with their ADLS paths
tables = {
    "supply_chain_100gb":
        f"{BASE_ABFSS}/silver/supply_chain_100gb",
    "pagerank_results":
        f"{BASE_ABFSS}/gold/pagerank_results",
    "high_risk_paths":
        f"{BASE_ABFSS}/gold/high_risk_paths",
    "risk_clusters_transactional":
        f"{BASE_ABFSS}/gold/risk_clusters_transactional",
    "risk_clusters_graph_augmented":
        f"{BASE_ABFSS}/gold/risk_clusters_graph_augmented",
    "feature_catalog":
        f"{BASE_ABFSS}/gold/feature_catalog",
    "feature_importances":
        f"{BASE_ABFSS}/gold/feature_importances",
    "feature_stability":
        f"{BASE_ABFSS}/gold/feature_stability",
    "scalability_benchmarks":
        f"{BASE_ABFSS}/gold/scalability_benchmarks",
}

print("Re-registering tables...\n")
for table_name, path in tables.items():
    try:
        # Check Delta files exist
        dbutils.fs.ls(path + "/_delta_log")

        # Drop and recreate
        spark.sql(
            f"DROP TABLE IF EXISTS "
            f"supply_chain_db.{table_name}")
        spark.sql(f"""
            CREATE TABLE supply_chain_db.{table_name}
            USING DELTA
            LOCATION '{path}'
        """)
        count = spark.table(
            f"supply_chain_db.{table_name}").count()
        print(f"  OK  {table_name:45s} {count:>15,} rows")

    except Exception as e:
        print(f"  --  {table_name:45s} "
              f"(not found — will be created later)")

print("\nAll available tables registered.")
print("\nCurrent tables in supply_chain_db:")
spark.sql("SHOW TABLES IN supply_chain_db") \
     .show(truncate=False)

Re-registering tables...

  OK  supply_chain_100gb                                101,840,453 rows
  --  pagerank_results                              (not found — will be created later)
  --  high_risk_paths                               (not found — will be created later)
  OK  risk_clusters_transactional                               561 rows
  OK  risk_clusters_graph_augmented                             561 rows
  OK  feature_catalog                                            10 rows
  OK  feature_importances                                         5 rows
  --  feature_stability                             (not found — will be created later)
  --  scalability_benchmarks                        (not found — will be created later)

All available tables registered.

Current tables in supply_chain_db:
+---------------+-----------------------------+-----------+
|database       |tableName                    |isTemporary|
+---------------+-----------------------------+-----------+
|supply

In [0]:


from graphframes import GraphFrame
from pyspark.sql import functions as F
import mlflow, time

STORAGE_ACCOUNT     = "adlssupplychain"
CONTAINER           = "supplychain-data"
BASE_ABFSS          = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD_PAGERANK_ABFSS = f"{BASE_ABFSS}/gold/pagerank_results"
GOLD_MOTIF_ABFSS    = f"{BASE_ABFSS}/gold/high_risk_paths"
CHECKPOINT_ABFSS    = f"{BASE_ABFSS}/checkpoints/pagerank_temp"

mlflow.set_experiment("/supply-chain-pipeline")


spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set(
    "spark.databricks.delta.overwriteSchema.enabled",  "true")
print("Delta schema config set.")

spark.sql("DROP TABLE IF EXISTS supply_chain_db.pagerank_results")
spark.sql("DROP TABLE IF EXISTS supply_chain_db.high_risk_paths")

for path in [GOLD_PAGERANK_ABFSS, GOLD_MOTIF_ABFSS,
             CHECKPOINT_ABFSS]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Cleared: {path.split('/')[-1]}")
    except Exception:
        print(f"  Already empty: {path.split('/')[-1]}")

spark.sql("CLEAR CACHE")
spark.catalog.clearCache()
print("Clean slate confirmed.\n")

Delta schema config set.
  Cleared: pagerank_results
  Cleared: high_risk_paths
  Cleared: pagerank_temp
Clean slate confirmed.



In [0]:

# SECTION 1 — Load and clean

print("Loading silver table...")
t_load = time.time()

df = (spark.table("supply_chain_db.supply_chain_100gb")
      .withColumn("Order_Region",
          F.trim(F.col("Order_Region")))
      .withColumn("Market",
          F.trim(F.col("Market")))
      .withColumn("Order_Country",
          F.trim(F.col("Order_Country")))
      .withColumn("Customer_City",
          F.trim(F.col("Customer_City")))
      .withColumn("Customer_Country",
          F.trim(F.col("Customer_Country")))
     ).cache()

row_count = df.count()
print(f"Loaded {row_count:,} rows in {time.time()-t_load:.0f}s")


Loading silver table...
Loaded 101,840,453 rows in 259s


In [0]:

# SECTION 2 — Hub aggregates (vertices)

print("\nBuilding hub aggregates...")

hubs_agg = (df
    .groupBy("Market", "Order_Region", "Order_Country")
    .agg(
        F.avg("Late_delivery_risk")
         .cast("double").alias("avg_delay_risk"),
        F.count("Order_Id")
         .cast("long").alias("order_count"),
        F.avg("Sales")
         .cast("double").alias("avg_sales"),
        F.avg("Benefit_per_order")
         .cast("double").alias("avg_benefit"),
        F.avg("Order_Profit_Per_Order")
         .cast("double").alias("avg_profit"),
        F.avg("Order_Item_Discount_Rate")
         .cast("double").alias("avg_discount_rate"),
        F.sum("Sales")
         .cast("double").alias("total_sales"),
        F.avg("Days_for_shipping_real")
         .cast("double").alias("avg_ship_days"),
        F.avg(
            F.col("Days_for_shipping_real") -
            F.col("Days_for_shipment_scheduled")
        ).cast("double").alias("avg_delay_gap"),
        F.countDistinct("Customer_Id")
         .cast("long").alias("unique_customers"),
    )
    .withColumn("id",
        F.concat_ws("|",
            F.col("Market"),
            F.col("Order_Region"),
            F.col("Order_Country")))
).cache()

v_count = hubs_agg.count()
print(f"Hub vertices: {v_count:,}")

hub_vertices = hubs_agg.select(
    "id", "Market", "Order_Region", "Order_Country",
    "avg_delay_risk", "order_count", "avg_sales",
    "avg_benefit", "avg_profit", "avg_discount_rate",
    "total_sales", "avg_ship_days", "avg_delay_gap",
    "unique_customers",
)


Building hub aggregates...
Hub vertices: 167


In [0]:

# SECTION 3 — Build REAL edges from customer order flows
#
# METHODOLOGY: Two hubs are connected if real customers
# placed orders at both hubs. This is derived directly from
# actual order records — not from statistical thresholds.
#
# Edge weight = number of shared customers (flow volume)
# Edge risk   = avg delay risk of source hub
#
# Shared customers indicate genuine supply chain dependency —
# the same customer base relies on both hubs, so disruption
# in one hub directly impacts the other hub's customer service.

print("\nBuilding REAL edges from customer order flows...")
print("Approach: two hubs share an edge if real customers")
print("placed orders at both — derived from actual order data.")

# Step 1: Get unique customer→hub mappings from actual orders
cust_hub = (df
    .select(
        F.col("Customer_Id").cast("string").alias("customer_id"),
        F.concat_ws("|",
            F.col("Market"),
            F.col("Order_Region"),
            F.col("Order_Country")).alias("hub_id"),
        F.col("Late_delivery_risk").cast("double"),
        F.col("Sales").cast("double"),
    )
    .distinct()
).cache()

total_cust_hub = cust_hub.count()
print(f"Unique customer-hub pairs: {total_cust_hub:,}")

# Step 2: Customers in multiple hubs
multi_hub_customers = (cust_hub
    .groupBy("customer_id")
    .agg(F.countDistinct("hub_id").alias("n_hubs"))
    .filter(F.col("n_hubs") > 1)
)
multi_count = multi_hub_customers.count()
print(f"Customers ordering from 2+ hubs: {multi_count:,}")
print(f"  (These customers create real inter-hub dependencies)")

# Step 3: Self-join on customer to create hub→hub edges
# If customer C orders from hub A AND hub B,
# then A→B and B→A are real supply chain links
shared_customer_edges = (cust_hub.alias("a")
    .join(
        cust_hub.alias("b"),
        on=(
            (F.col("a.customer_id") == F.col("b.customer_id")) &
            (F.col("a.hub_id")      != F.col("b.hub_id"))
        ),
        how="inner"
    )
    .groupBy(
        F.col("a.hub_id").alias("src"),
        F.col("b.hub_id").alias("dst"),
    )
    .agg(
        # Weight = shared customer count (actual flow volume)
        F.countDistinct("a.customer_id")
         .cast("double").alias("weight"),
        # Edge risk based on source hub delay profile
        F.avg("a.Late_delivery_risk")
         .cast("double").alias("edge_risk_raw"),
        F.avg("a.Sales")
         .cast("double").alias("avg_sales_on_edge"),
    )
    .withColumn(
        "edge_risk",
        F.when(F.col("edge_risk_raw") > 0.5, 1)
         .otherwise(0)
    )
    .withColumn(
        "shipping_mode",
        F.lit("shared_customer_flow")
    )
    .drop("edge_risk_raw")
)

edge_count = shared_customer_edges.count()
print(f"\nReal hub→hub edges from shared customers: {edge_count:,}")
print("(Each edge = real customers ordering from both hubs)")

# Verify edges reference valid hub IDs
hub_id_set  = hub_vertices.select("id").distinct()
invalid_src = shared_customer_edges.join(
    hub_id_set,
    shared_customer_edges.src == hub_id_set.id,
    "left_anti").count()
invalid_dst = shared_customer_edges.join(
    hub_id_set.withColumnRenamed("id", "id2"),
    shared_customer_edges.dst == F.col("id2"),
    "left_anti").count()
print(f"\nEdge validity check:")
print(f"  Invalid src: {invalid_src}  (must be 0)")
print(f"  Invalid dst: {invalid_dst}  (must be 0)")
if invalid_src > 0 or invalid_dst > 0:
    raise ValueError("Edge validation failed.")

# Edge weight distribution
print("\nShared customer count distribution:")
shared_customer_edges.select(
    F.min("weight").alias("min_shared_customers"),
    F.max("weight").alias("max_shared_customers"),
    F.avg("weight").alias("avg_shared_customers"),
    F.count("*").alias("total_edges"),
).show()

# Top hub connections by shared customer volume
print("Top 10 strongest hub connections:")
shared_customer_edges \
    .orderBy(F.col("weight").desc()) \
    .select("src", "dst",
            F.round("weight", 0).alias("shared_customers"),
            "edge_risk") \
    .show(10, truncate=False)



Building REAL edges from customer order flows...
Approach: two hubs share an edge if real customers
placed orders at both — derived from actual order data.
Unique customer-hub pairs: 101,840,453
Customers ordering from 2+ hubs: 6,971,258
  (These customers create real inter-hub dependencies)

Real hub→hub edges from shared customers: 10,738
(Each edge = real customers ordering from both hubs)

Edge validity check:
  Invalid src: 0  (must be 0)
  Invalid dst: 0  (must be 0)

Shared customer count distribution:
+--------------------+--------------------+--------------------+-----------+
|min_shared_customers|max_shared_customers|avg_shared_customers|total_edges|
+--------------------+--------------------+--------------------+-----------+
|               600.0|            567000.0|  11722.322406407153|      10738|
+--------------------+--------------------+--------------------+-----------+

Top 10 strongest hub connections:
+-------------------------------+-------------------------------

In [0]:

# SECTION 4 — Build GraphFrame + PageRank

print("\nBuilding GraphFrame with real order-flow edges...")
g = GraphFrame(hub_vertices, shared_customer_edges)

with mlflow.start_run(run_name="graph_analytics_real_edges"):
    mlflow.set_tag("team_member",   "Sudheesh U")
    mlflow.set_tag("compute_type",  "Standard_D4s_v3")
    mlflow.set_tag("edge_method",   "shared_customer_flow")
    mlflow.log_param("pagerank_reset",        0.15)
    mlflow.log_param("pagerank_maxiter",      10)
    mlflow.log_param("vertex_count",          v_count)
    mlflow.log_param("edge_count",            edge_count)
    mlflow.log_param("multi_hub_customers",   multi_count)
    mlflow.log_param("total_cust_hub_pairs",  total_cust_hub)

    t0 = time.time()

    #  PageRank ─
    print("Running PageRank on real order-flow graph...")
    t1      = time.time()
    pr      = g.pageRank(resetProbability=0.15, maxIter=10)
    pr_time = time.time() - t1
    print(f"PageRank done in {pr_time:.1f}s")
    mlflow.log_metric("pagerank_time_sec", pr_time)

    # Distribution check
    pr_stats = pr.vertices.select(
        F.min("pagerank").alias("min_pr"),
        F.max("pagerank").alias("max_pr"),
        F.avg("pagerank").alias("avg_pr"),
        F.count("*").alias("hub_count"),
        F.countDistinct("pagerank").alias("distinct_scores"),
    ).collect()[0]

    min_pr          = float(pr_stats["min_pr"])
    max_pr          = float(pr_stats["max_pr"])
    distinct_scores = int(pr_stats["distinct_scores"])
    pr_range        = max_pr - min_pr \
                      if abs(max_pr - min_pr) > 1e-9 else 1.0

    print(f"\nPageRank distribution (real edges):")
    print(f"  Min            : {min_pr:.6f}")
    print(f"  Max            : {max_pr:.6f}")
    print(f"  Avg            : {float(pr_stats['avg_pr']):.6f}")
    print(f"  Ratio (max/min): {max_pr/max(min_pr,1e-9):.2f}x")
    print(f"  Distinct scores: {distinct_scores}")
    print(f"  (Higher distinct scores = more realistic topology)")

    mlflow.log_metric("pagerank_min",            min_pr)
    mlflow.log_metric("pagerank_max",            max_pr)
    mlflow.log_metric("pagerank_ratio",
                      max_pr / max(min_pr, 1e-9))
    mlflow.log_metric("pagerank_distinct_scores",distinct_scores)

    #  Build final DataFrame 
    pr_scores_only = (pr.vertices
        .select(
            F.col("id"),
            F.col("pagerank").cast("double").alias("pagerank_raw")
        )
    )

    pr_final = (hub_vertices
        .join(pr_scores_only, on="id", how="left")
        .withColumn(
            "pagerank",
            ((F.col("pagerank_raw") - min_pr) / pr_range)
             .cast("double")
        )
        .fillna({"pagerank_raw": min_pr, "pagerank": 0.0})
        .select(
            F.col("id"),
            F.col("Market"),
            F.col("Order_Region"),
            F.col("Order_Country"),
            F.col("avg_delay_risk").cast("double"),
            F.col("order_count").cast("long"),
            F.col("avg_sales").cast("double"),
            F.col("avg_benefit").cast("double"),
            F.col("avg_profit").cast("double"),
            F.col("avg_discount_rate").cast("double"),
            F.col("total_sales").cast("double"),
            F.col("avg_ship_days").cast("double"),
            F.col("avg_delay_gap").cast("double"),
            F.col("unique_customers").cast("long"),
            F.col("pagerank_raw").cast("double"),
            F.col("pagerank").cast("double"),
        )
    )

    print("\nTop 20 bottleneck hubs by PageRank centrality:")
    display(
        pr_final
        .orderBy(F.col("pagerank").desc())
        .select(
            "id", "Order_Region", "Market",
            F.round("avg_delay_risk", 3).alias("delay_risk"),
            "order_count", "unique_customers",
            F.round("pagerank_raw", 6).alias("pr_raw"),
            F.round("pagerank", 6).alias("pr_norm"),
        )
        .limit(20)
    )

    # Checkpoint to break GraphFrame schema lineage
    print("\nCheckpointing...")
    sc = spark.sparkContext
    sc._jsc.hadoopConfiguration().set(
        f"fs.azure.account.key.{STORAGE_ACCOUNT}"
        f".dfs.core.windows.net",
        dbutils.secrets.get(
            scope="adls-scope",
            key="adls-account-key").strip()
    )
    spark.sparkContext.setCheckpointDir(CHECKPOINT_ABFSS)
    pr_checkpointed = pr_final.checkpoint(eager=True)
    chk_count = pr_checkpointed.count()
    print(f"Checkpoint complete. Rows: {chk_count:,}")

    # Write via saveAsTable
    print("Writing PageRank to Delta...")
    spark.sql(
        "DROP TABLE IF EXISTS supply_chain_db.pagerank_results")
    (pr_checkpointed
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("mergeSchema",     "true")
        .option("path", GOLD_PAGERANK_ABFSS)
        .saveAsTable("supply_chain_db.pagerank_results"))
    print("PageRank saved.")

    try:
        dbutils.fs.rm(CHECKPOINT_ABFSS, recurse=True)
    except Exception:
        pass

    # Verify
    saved_pr = spark.table("supply_chain_db.pagerank_results")
    print("\nNormalised PageRank range:")
    saved_pr.select(
        F.min("pagerank").alias("min"),
        F.max("pagerank").alias("max"),
        F.avg("pagerank").alias("avg"),
        F.countDistinct("pagerank").alias("distinct_scores"),
        F.count("*").alias("total_hubs"),
    ).show()

    mlflow.log_metric("saved_hub_count", chk_count)



Building GraphFrame with real order-flow edges...
Running PageRank on real order-flow graph...
PageRank done in 159.6s

PageRank distribution (real edges):
  Min            : 0.163816
  Max            : 2.437714
  Avg            : 1.000000
  Ratio (max/min): 14.88x
  Distinct scores: 167
  (Higher distinct scores = more realistic topology)

Top 20 bottleneck hubs by PageRank centrality:


id,Order_Region,Market,delay_risk,order_count,unique_customers,pr_raw,pr_norm
LATAM|Central America|México,Central America,LATAM,0.538,7476622,2167828,2.437714,1.0
LATAM|South America|Brasil,South America,LATAM,0.53,4539714,1395000,2.389042,0.978595
Europe|Western Europe|Francia,Western Europe,Europe,0.539,7448491,2452506,2.365332,0.968168
Europe|Northern Europe|Reino Unido,Northern Europe,Europe,0.523,4141866,1496400,2.215692,0.90236
Pacific Asia|Oceania|Australia,Oceania,Pacific Asia,0.528,4777258,2047227,2.166675,0.880804
USCA|West of USA|Estados Unidos,West of USA,USCA,0.524,4495009,1397870,2.147093,0.872192
Europe|Western Europe|Alemania,Western Europe,Europe,0.55,5399358,1850400,2.127234,0.863459
USCA|East of USA|Estados Unidos,East of USA,USCA,0.54,3888094,1235399,2.104881,0.853629
Europe|Southern Europe|Italia,Southern Europe,Europe,0.518,2774177,1019429,2.063652,0.835498
LATAM|Caribbean|República Dominicana,Caribbean,LATAM,0.537,2080707,693000,2.057085,0.832609



Checkpointing...
Checkpoint complete. Rows: 167
Writing PageRank to Delta...
PageRank saved.

Normalised PageRank range:
+---+---+-------------------+---------------+----------+
|min|max|                avg|distinct_scores|total_hubs|
+---+---+-------------------+---------------+----------+
|0.0|1.0|0.36773154321700124|            167|       167|
+---+---+-------------------+---------------+----------+



In [0]:

    # SECTION 5 — Motif Discovery
    # Pattern: hub_A → hub_B → hub_C (genuine transshipment)
    # With real edges: this finds hubs where shared customer
    # flows create triangular dependency chains
    # Self-loops excluded: origin must differ from destination
    

    print("\nRunning Motif Discovery...")
    print("Finding triangular high-risk customer flow chains...")
    t2     = time.time()
    motifs = g.find("(a)-[e1]->(b); (b)-[e2]->(c); !(a)-[]->(c)")

    high_risk_paths = (motifs
        .filter(
            (F.col("e1.edge_risk") > 0) &
            (F.col("e2.edge_risk") > 0) &
            # Exclude self-loops (origin == destination)
            (F.col("a.id") != F.col("c.id"))
        )
        .select(
            F.col("a.id").alias("origin"),
            F.col("b.id").alias("transshipment_hub"),
            F.col("c.id").alias("destination"),
            F.col("e1.weight").cast("double")
             .alias("shared_customers_leg1"),
            F.col("e2.weight").cast("double")
             .alias("shared_customers_leg2"),
            F.col("e1.shipping_mode").alias("leg1_type"),
            F.col("e2.shipping_mode").alias("leg2_type"),
            (F.col("e1.edge_risk") +
             F.col("e2.edge_risk")).alias("combined_risk"),
            F.col("a.avg_delay_risk").alias("origin_risk"),
            F.col("b.avg_delay_risk").alias("hub_risk"),
            F.col("c.avg_delay_risk").alias("dest_risk"),
            # Total shared customer exposure
            (F.col("e1.weight") +
             F.col("e2.weight")).alias("total_customer_exposure"),
        )
        .orderBy(
            F.col("combined_risk").desc(),
            F.col("total_customer_exposure").desc()
        )
    )

    motif_count = high_risk_paths.count()
    motif_time  = time.time() - t2
    print(f"Motif done in {motif_time:.1f}s")
    print(f"Genuine triangular risk chains: {motif_count:,}")

    mlflow.log_metric("motif_time_sec",        motif_time)
    mlflow.log_metric("high_risk_paths_count", motif_count)

    if motif_count > 0:
        print("\nTop 15 high-risk transshipment chains:")
        display(high_risk_paths.limit(15))

        print("\nRiskiest transshipment hubs:")
        high_risk_paths \
            .groupBy("transshipment_hub") \
            .agg(
                F.count("*").alias("times_as_hub"),
                F.avg("hub_risk").alias("avg_hub_risk"),
                F.sum("total_customer_exposure")
                 .alias("total_exposure"),
            ) \
            .orderBy(F.desc("times_as_hub")) \
            .show(10, truncate=False)
    else:
        print("No genuine triangular risk chains found.")
        print("Valid result — saving empty table.")

    spark.sql(
        "DROP TABLE IF EXISTS supply_chain_db.high_risk_paths")
    (high_risk_paths
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("mergeSchema",     "true")
        .option("path", GOLD_MOTIF_ABFSS)
        .saveAsTable("supply_chain_db.high_risk_paths"))
    print("High-risk paths saved.")

    #  Summary 
    total_time = time.time() - t0
    mlflow.log_metric("total_graph_time_sec", total_time)

    print("\n" + "="*60)
    print("GRAPH ANALYTICS COMPLETE")
    print("="*60)
    print(f"  Hub vertices      : {v_count:,}")
    print(f"  Real edges        : {edge_count:,}")
    print(f"  Multi-hub custs   : {multi_count:,}")
    print(f"  PageRank time     : {pr_time:.0f}s")
    print(f"  PR min (raw)      : {min_pr:.4f}")
    print(f"  PR max (raw)      : {max_pr:.4f}")
    print(f"  Centrality ratio  : {max_pr/max(min_pr,1e-9):.2f}x")
    print(f"  Distinct scores   : {distinct_scores}")
    print(f"  High-risk chains  : {motif_count:,}")
    print(f"  Total time        : {total_time:.0f}s "
          f"({total_time/60:.1f} min)")
    print()
    print("  Thesis finding:")
    print(f"  Graph built from {multi_count:,} customers who")
    print(f"  ordered from multiple hubs — real supply chain")
    print(f"  dependencies, not mathematical thresholds.")
    print(f"  Top hub is {max_pr/max(min_pr,1e-9):.1f}x more central")
    print(f"  than least central hub across {distinct_scores}")
    print(f"  distinct centrality tiers.")

df.unpersist()
hubs_agg.unpersist()
cust_hub.unpersist()
print("\nNotebook 03 complete. Proceed to Notebook 04.")


Running Motif Discovery...
Finding triangular high-risk customer flow chains...
Motif done in 185.4s
Genuine triangular risk chains: 338,621

Top 15 high-risk transshipment chains:


origin,transshipment_hub,destination,shared_customers_leg1,shared_customers_leg2,leg1_type,leg2_type,combined_risk,origin_risk,hub_risk,dest_risk,total_customer_exposure
Europe|Western Europe|Alemania,LATAM|Central America|México,Europe|Western Europe|Francia,438600.0,567000.0,shared_customer_flow,shared_customer_flow,2,0.5503467264071025,0.53751814656405,0.5394404047746046,1005600.0
Europe|Western Europe|Francia,LATAM|Central America|México,Europe|Western Europe|Alemania,567000.0,438600.0,shared_customer_flow,shared_customer_flow,2,0.5394404047746046,0.53751814656405,0.5503467264071025,1005600.0
Europe|Western Europe|Francia,LATAM|Central America|México,LATAM|South America|Brasil,567000.0,433200.0,shared_customer_flow,shared_customer_flow,2,0.5394404047746046,0.53751814656405,0.529722136680857,1000200.0
LATAM|South America|Brasil,LATAM|Central America|México,Europe|Western Europe|Francia,433200.0,567000.0,shared_customer_flow,shared_customer_flow,2,0.529722136680857,0.53751814656405,0.5394404047746046,1000200.0
Europe|Western Europe|Alemania,Europe|Western Europe|Francia,LATAM|Central America|México,407400.0,567000.0,shared_customer_flow,shared_customer_flow,2,0.5503467264071025,0.5394404047746046,0.53751814656405,974400.0
LATAM|Central America|México,Europe|Western Europe|Francia,Europe|Western Europe|Alemania,567000.0,407400.0,shared_customer_flow,shared_customer_flow,2,0.53751814656405,0.5394404047746046,0.5503467264071025,974400.0
Europe|Western Europe|Francia,LATAM|Central America|México,USCA|West of USA|Estados Unidos,567000.0,404400.0,shared_customer_flow,shared_customer_flow,2,0.5394404047746046,0.53751814656405,0.5236374387681982,971400.0
USCA|West of USA|Estados Unidos,LATAM|Central America|México,Europe|Western Europe|Francia,404400.0,567000.0,shared_customer_flow,shared_customer_flow,2,0.5236374387681982,0.53751814656405,0.5394404047746046,971400.0
LATAM|South America|Brasil,Europe|Western Europe|Francia,LATAM|Central America|México,396600.0,567000.0,shared_customer_flow,shared_customer_flow,2,0.529722136680857,0.5394404047746046,0.53751814656405,963600.0
LATAM|Central America|México,Europe|Western Europe|Francia,LATAM|South America|Brasil,567000.0,396600.0,shared_customer_flow,shared_customer_flow,2,0.53751814656405,0.5394404047746046,0.529722136680857,963600.0



Riskiest transshipment hubs:


+----------------------------------+------------+------------------+--------------+
|transshipment_hub                 |times_as_hub|avg_hub_risk      |total_exposure|
+----------------------------------+------------+------------------+--------------+
|LATAM|Central America|México      |10679       |0.5375181465640425|1.403906624E9 |
|Europe|Western Europe|Francia     |9107        |0.539440404774588 |1.26309387E9  |
|Europe|Western Europe|Alemania    |9035        |0.5503467264071042|9.29366117E8  |
|Europe|Northern Europe|Reino Unido|8951        |0.5230932145076631|7.87572E8     |
|LATAM|South America|Brasil        |8902        |0.5297221366808677|7.7324703E8   |
|USCA|East of USA|Estados Unidos   |8498        |0.5398261976176563|6.71838744E8  |
|USCA|West of USA|Estados Unidos   |7956        |0.5236374387682106|7.21224335E8  |
|Pacific Asia|Oceania|Australia    |7870        |0.5284736558084216|7.4377641E8   |
|LATAM|Central America|Nicaragua   |7345        |0.5535065987180908|3.265252